# Dormancy zoo: Colab launcher (Phase 4b)

Plan: `reports/2026-11-14-colab-plan.md`, approved by the PI on 2026-11-14.
**Launcher only.** This notebook holds no training code. It clones a pinned commit, installs the
environment with pixi, and runs one queue with `python -m zoo.colab queue`. Re-running it resumes
the queue: finished runs are skipped and interrupted training resumes from `resume.pt`.

Rules (AGENTS.md, "Colab"):
- The GitHub token comes only from Colab secrets (`GITHUB_TOKEN`), through a git credential helper.
  It is never printed, never in a URL, never written to `.git/config`, never committed.
- Weights go only to Drive. Git receives only small result files (`results/colab/<queue>/`,
  `results/manifest_colab.csv`), on a feature branch.
- The first queue is the equivalence test. Nothing else runs until the PI's OK.

Set the four parameters in the next cell, then *Runtime → Run all*. Use a GPU runtime.

In [ ]:
# ---- parameters (the agent supplies COMMIT and QUEUE; the PI supplies REPO) ----
REPO = "github.com/OWNER/REPO.git"          # private repository, without https:// and without any token
COMMIT = "PINNED_COMMIT_SHA"                 # full 40-character SHA
QUEUE = "configs/colab/queue_equiv.yaml"     # first run: the equivalence test only
DRIVE_DIR = "/content/drive/MyDrive/dormancy_zoo"
PIXI_VERSION = "v0.81.0"                     # same pixi as on the laptop
PIXI_MANIFEST = "pixi.toml"                  # laptop lock; a separate Colab environment only if the driver needs it
PUSH_RESULTS = True

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
print(sorted(os.listdir(f"{DRIVE_DIR}/runs"))[:40])

In [ ]:
# Clone the pinned commit. The token is passed to git through GIT_ASKPASS from an environment
# variable of the child process only; output is captured so nothing can echo it.
import os, subprocess, stat
from google.colab import userdata

ASKPASS = "/content/.git_askpass.sh"
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\nprintf "%s" "$ZOO_GH_TOKEN"\n')
os.chmod(ASKPASS, stat.S_IRWXU)

def git(*args, cwd="/content/dormancy_zoo", secret=False):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    if secret:
        env.update(GIT_ASKPASS=ASKPASS, ZOO_GH_TOKEN=userdata.get("GITHUB_TOKEN"))
    r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}); stderr withheld in case it holds secrets")
    return r.stdout.strip()

if not os.path.exists("/content/dormancy_zoo/.git"):
    git("clone", "--quiet", f"https://x-access-token@{REPO}", "/content/dormancy_zoo", cwd="/content", secret=True)
git("fetch", "--quiet", "origin", secret=True)
git("checkout", "--quiet", COMMIT)
head = git("rev-parse", "HEAD")
assert head == COMMIT, f"HEAD {head} is not the pinned commit"
print("at pinned commit", head)

In [ ]:
# Environment: pixi from the lock file, then check CUDA. If CUDA is unavailable, stop and send the
# printed driver version to the agent (a separate Colab environment would be defined; the laptop lock is never changed).
import os, subprocess
os.chdir("/content/dormancy_zoo")
if not os.path.exists(os.path.expanduser("~/.pixi/bin/pixi")):
    subprocess.run(f"curl -fsSL https://pixi.sh/install.sh | PIXI_VERSION={PIXI_VERSION} bash", shell=True, check=True)
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]
subprocess.run(["pixi", "install", "--frozen", "--manifest-path", PIXI_MANIFEST], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
r = subprocess.run(["pixi", "run", "--manifest-path", PIXI_MANIFEST, "python", "-c",
                    "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
assert r.stdout.strip().endswith("True"), "CUDA not available with this environment: stop and report the driver version above"


In [ ]:
# The PI's upload: every teacher file present on Drive and matching the laptop's hashes.
import subprocess
os.environ["ZOO_ENV"] = "colab"
subprocess.run(["pixi", "run", "--manifest-path", PIXI_MANIFEST, "python", "-m", "zoo.colab", "verify",
                f"{DRIVE_DIR}/upload_files.sha256", "--root", f"{DRIVE_DIR}/runs"], check=True)

In [ ]:
# Run the queue (resumable). Finished runs are copied to Drive and re-hashed there after every step.
import subprocess, sys
p = subprocess.Popen(["pixi", "run", "--manifest-path", PIXI_MANIFEST, "python", "-u", "-m", "zoo.colab", "queue",
                      QUEUE, "--drive", DRIVE_DIR], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if "Loading weights" not in line and "UNEXPECTED" not in line:
        sys.stdout.write(line)
assert p.wait() == 0, "queue did not finish: re-run the notebook to resume"


In [ ]:
# Push only the small result files to a feature branch, after guards.
import os, re, pathlib, datetime
if PUSH_RESULTS:
    qname = pathlib.Path(QUEUE).stem
    branch = f"colab/{qname}-{datetime.date.today():%Y%m%d}"
    name, email = git("log", "-1", "--format=%an"), git("log", "-1", "--format=%ae")
    git("config", "user.name", name); git("config", "user.email", email)
    git("checkout", "--quiet", "-B", branch)
    git("add", f"results/colab/{qname}", "results/manifest_colab.csv")
    staged = [s for s in git("diff", "--cached", "--name-only").splitlines() if s]
    token = userdata.get("GITHUB_TOKEN")
    for s in staged:
        p = pathlib.Path(s)
        assert s.startswith(f"results/colab/{qname}/") or s == "results/manifest_colab.csv", f"refusing {s}"
        assert p.suffix in {".csv", ".json", ".yaml", ".sha256"} and p.stat().st_size <= 2 * 1024**2, f"refusing {s}"
        assert not s.startswith("runs/") and p.suffix not in {".safetensors", ".pt", ".bin"}, f"refusing {s}"
        txt = p.read_text(errors="ignore")
        assert token not in txt and not re.search(r"(ghp_|github_pat_)[A-Za-z0-9_]{20,}", txt), f"token pattern in {s}"
    del token
    git("commit", "--quiet", "-m", f"Colab results: {qname} (commit {COMMIT[:12]})")
    git("push", "--quiet", "origin", f"HEAD:refs/heads/{branch}", secret=True)
    print(f"pushed {len(staged)} files to {branch}")